# Amazon Review Sentiment Analysis — Model Training

## What this notebook does
Trains and compares sentiment classifiers systematically across multiple configurations.

## Experiments performed
1. N-gram comparison: unigram / bigram / trigram / uni+bi / uni+bi+tri
2. RegParam grid search: 0.001 / 0.003 / 0.005 / 0.01
3. Sublinear TF scaling vs raw TF
4. Helpfulness-weighted training vs uniform weights
5. Final model comparison: LR (best config) vs OneVsRest SVC vs Naive Bayes

## Why this matters for the presentation
Instead of showing one model and one result, this notebook demonstrates a rigorous
experimental process: we tested N configurations, measured each one, and chose the
best based on evidence. This is what real ML pipelines look like.

## Cell 1 — Spark Session

Java 17 requires explicit module-access flags for Spark's internal reflection.
We set these before any PySpark import — JVM reads JAVA_HOME at startup.

- `driver.memory = 6g` — safe ceiling for 8GB free RAM
- `setLogLevel(ERROR)` — suppresses WARN/INFO noise in notebook output

In [1]:
import os
import json
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

os.environ['JAVA_HOME'] = '/opt/homebrew/opt/openjdk@17'
os.environ['PATH']      = '/opt/homebrew/opt/openjdk@17/bin:' + os.environ.get('PATH', '')

from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF, StringIndexer
from pyspark.ml.classification import (
    LogisticRegression, LinearSVC, OneVsRest, NaiveBayes
)
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

_jvm_opts = ' '.join([
    '--add-opens=java.base/java.lang=ALL-UNNAMED',
    '--add-opens=java.base/java.lang.invoke=ALL-UNNAMED',
    '--add-opens=java.base/java.lang.reflect=ALL-UNNAMED',
    '--add-opens=java.base/java.io=ALL-UNNAMED',
    '--add-opens=java.base/java.net=ALL-UNNAMED',
    '--add-opens=java.base/java.nio=ALL-UNNAMED',
    '--add-opens=java.base/java.util=ALL-UNNAMED',
    '--add-opens=java.base/java.util.concurrent=ALL-UNNAMED',
    '--add-opens=java.base/sun.nio.ch=ALL-UNNAMED',
    '--add-opens=java.base/sun.nio.cs=ALL-UNNAMED',
    '--add-opens=java.base/sun.security.action=ALL-UNNAMED',
])

spark = (
    SparkSession.builder
    .master('local[2]')
    .appName('AmazonSentiment_Experiments')
    .config('spark.driver.memory', '6g')
    .config('spark.executor.memory', '6g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.driver.extraJavaOptions', _jvm_opts)
    .config('spark.executor.extraJavaOptions', _jvm_opts)
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')

print(f'Spark version : {spark.version}')
print(f'Java home     : {os.environ["JAVA_HOME"]}')


26/05/08 12:01:12 WARN Utils: Your hostname, Louays-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.1.25 instead (on interface en0)
26/05/08 12:01:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/08 12:01:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version : 3.5.1
Java home     : /opt/homebrew/opt/openjdk@17


## Cell 2 — Load Data

Loads train / val / test CSVs produced by 01_eda_preprocessing.ipynb.
The `helpfulness_weight` column is now included — used during LR training.

`repartition(4)` distributes data across 8 parallel Spark tasks.


In [2]:
import numpy as np

BASE = '/Users/beethoven/BigData/AmazonReview'
COLS = ['clean_text', 'sentiment', 'ProductId', 'UserId', 'Id', 'Time', 'helpfulness_weight']

def load(path):
    df = pd.read_csv(path)
    if 'helpfulness_weight' not in df.columns:
        df['helpfulness_weight'] = np.log1p(df['HelpfulnessNumerator']).clip(lower=1.0)
    return (
        spark
        .createDataFrame(df[COLS].dropna())
        .repartition(2)
    )

train = load(f'{BASE}/data/train.csv')
val   = load(f'{BASE}/data/val.csv')
test  = load(f'{BASE}/data/test.csv')

print(f'Train : {train.count():,}')
print(f'Val   : {val.count():,}')
print(f'Test  : {test.count():,}')
print('\nClass distribution in training set:')
train.groupBy('sentiment').count().orderBy('count', ascending=False).show()


Train : 314,900
Val   : 39,319
Test  : 39,358

Class distribution in training set:


+---------+------+
|sentiment| count|
+---------+------+
| positive|245435|
| negative| 45659|
|  neutral| 23806|
+---------+------+



## Cell 3 — Handle Class Imbalance

Two strategies to handle the 78% / 14% / 8% positive / negative / neutral split:

### Strategy 1 — Class Weights
Formula: weight = total / (num_classes x class_count)
Neutral gets ~4.4x more penalty weight than positive.
Used by Logistic Regression which supports weightCol.

### Strategy 2 — Helpfulness-Boosted Weights
Same as class weights but multiplied by log(1 + helpful_votes).
Reviews that many users found helpful are given more training influence.
This is the new improvement over the previous version.

### Strategy 3 — Balanced Undersampling
Caps every class at the size of the smallest class (neutral).
Used by LinearSVC and NaiveBayes which do not support weightCol.

In [3]:
# Compute class weights
label_counts = train.groupBy('sentiment').count().collect()
total_count  = sum(r['count'] for r in label_counts)
num_classes  = len(label_counts)

weight_map = {
    r['sentiment']: total_count / (num_classes * r['count'])
    for r in label_counts
}
print('Class weights:')
for s, w in sorted(weight_map.items()):
    print(f'  {s:10s} -> {w:.4f}')

# Strategy 1: pure class weights
weights_df = spark.createDataFrame(
    [(k, float(v)) for k, v in weight_map.items()],
    ['sentiment', 'classWeight']
)
train_w = train.join(weights_df, on='sentiment', how='left')

# Strategy 2: class weights boosted by helpfulness
# Multiply class weight by helpfulness_weight — helpful reviews count more
train_wh = train_w.withColumn(
    'classWeight',
    F.col('classWeight') * F.col('helpfulness_weight')
)

# Strategy 3: balanced undersampling for SVC and NaiveBayes
min_count     = min(r['count'] for r in label_counts)
balanced_train = (
    train.filter(F.col('sentiment') == 'positive').limit(min_count)
    .union(train.filter(F.col('sentiment') == 'negative').limit(min_count))
    .union(train.filter(F.col('sentiment') == 'neutral' ).limit(min_count))
    .repartition(2)
)
print(f'\nBalanced sample: {min_count:,} per class -> {balanced_train.count():,} total')


Class weights:
  negative   -> 2.2989
  neutral    -> 4.4093
  positive   -> 0.4277



Balanced sample: 23,806 per class -> 71,418 total


## Cell 4 — Experiment 1: N-gram Configuration Comparison

We test 5 text feature configurations to find which n-gram combination gives best results.
All other parameters are held constant (LR with regParam=0.003, numFeatures=80000).

Configurations tested:
| Config | Description | What it captures |
|---|---|---|
| unigram | Individual words only | 'good', 'bad', 'excellent' |
| bigram | Word pairs only | 'not_good', 'highly_recommend' |
| trigram | Word triplets only | 'not_worth_money' |
| uni_bi | Words + pairs | Best of both |
| uni_bi_tri | Words + pairs + triplets | Maximum context |

Note: bigram-only and trigram-only configs typically underperform because they miss
single-word signals. The comparison table shows this empirically.

**Expected runtime: 25-35 minutes for all 5 configs.**

In [4]:
import time

# Fixed components shared across all experiments
indexer   = StringIndexer(inputCol='sentiment', outputCol='label', handleInvalid='keep')
evaluator = MulticlassClassificationEvaluator(
    labelCol='label', predictionCol='prediction', metricName='f1'
)

def make_tfidf_pipeline(input_col, num_features=80000, min_doc_freq=3):
    """Build TF-IDF pipeline from a given input column."""
    tokenizer  = Tokenizer(inputCol=input_col, outputCol='words')
    remover    = StopWordsRemover(inputCol='words', outputCol='filtered')
    hashingTF  = HashingTF(inputCol='filtered', outputCol='rawFeatures',
                           numFeatures=num_features)
    idf        = IDF(inputCol='rawFeatures', outputCol='features', minDocFreq=min_doc_freq)
    return [tokenizer, remover, hashingTF, idf, indexer]

def extract_ngrams(df_pd, config):
    """
    Extract only the requested n-gram types from clean_text.
    clean_text already contains unigrams_bigrams_trigrams concatenated.
    We split and take only the parts we want based on config.
    """
    def filter_text(text):
        tokens = str(text).split()
        unigrams  = [t for t in tokens if '_' not in t]
        bigrams   = [t for t in tokens if t.count('_') == 1]
        trigrams  = [t for t in tokens if t.count('_') == 2]
        if config == 'unigram':      return ' '.join(unigrams)
        if config == 'bigram':       return ' '.join(bigrams)
        if config == 'trigram':      return ' '.join(trigrams)
        if config == 'uni_bi':       return ' '.join(unigrams + bigrams)
        if config == 'uni_bi_tri':   return ' '.join(unigrams + bigrams + trigrams)
        return text
    return df_pd.apply(filter_text)

# Fixed LR for comparison
lr_fixed = LogisticRegression(
    maxIter=200, regParam=0.003, elasticNetParam=0.1,
    family='multinomial', weightCol='classWeight'
)

ngram_results = {}
configs = ['unigram', 'bigram', 'trigram', 'uni_bi', 'uni_bi_tri']

for config in configs:
    print(f'\nTesting config: {config}')
    t0 = time.time()

    # Filter train_w and val to use only this config's tokens
    train_pd = train_w.toPandas()
    val_pd   = val.toPandas()

    train_pd['ngram_text'] = extract_ngrams(train_pd['clean_text'], config)
    val_pd['ngram_text']   = extract_ngrams(val_pd['clean_text'],   config)

    train_sp = spark.createDataFrame(train_pd[['ngram_text', 'sentiment', 'classWeight']].dropna()).repartition(2)
    val_sp   = spark.createDataFrame(val_pd[['ngram_text', 'sentiment']].dropna()).repartition(2)

    try:
        stages   = make_tfidf_pipeline('ngram_text')
        pipeline = Pipeline(stages=stages + [lr_fixed])
        model    = pipeline.fit(train_sp)
        f1       = evaluator.evaluate(model.transform(val_sp))
        ngram_results[config] = round(f1, 4)
        print(f'  Val F1: {f1:.4f} ({time.time()-t0:.0f}s)')
    except Exception as e:
        print(f'  Failed: {e}')
        ngram_results[config] = None

# Print comparison table
print('\n' + '='*50)
print(' N-GRAM CONFIGURATION COMPARISON')
print('='*50)
print(f'{"Config":20s} | {"Val F1":>8s}')
print('-'*32)
for config in configs:
    score = ngram_results.get(config)
    score_str = f'{score:.4f}' if score else 'FAILED'
    marker = ' <-- BEST' if score == max(v for v in ngram_results.values() if v) else ''
    print(f'{config:20s} | {score_str:>8s}{marker}')

best_ngram = max(ngram_results, key=lambda k: ngram_results[k] or 0)
print(f'\nBest n-gram config: {best_ngram} (F1={ngram_results[best_ngram]:.4f})')



Testing config: unigram


  Val F1: 0.8279 (82s)

Testing config: bigram


  Val F1: 0.8134 (71s)

Testing config: trigram


  Val F1: 0.6609 (180s)

Testing config: uni_bi


  Val F1: 0.8560 (1046s)

Testing config: uni_bi_tri


  Val F1: 0.8494 (275s)

 N-GRAM CONFIGURATION COMPARISON
Config               |   Val F1
--------------------------------
unigram              |   0.8279
bigram               |   0.8134
trigram              |   0.6609
uni_bi               |   0.8560 <-- BEST
uni_bi_tri           |   0.8494

Best n-gram config: uni_bi (F1=0.8560)


## Cell 5 — Experiment 2: RegParam Grid Search

Regularization (regParam) controls how much the model is penalized for complexity.

- Too high (e.g. 0.1): model underfits — too simple, misses patterns
- Too low (e.g. 0.0001): model overfits — memorizes training data, fails on new reviews
- Sweet spot: typically 0.001 to 0.01 for TF-IDF text classification

We test 4 values systematically using the best n-gram config from Cell 4.
This is called a grid search — we vary one parameter while holding all others constant.

**Expected runtime: 20-30 minutes.**

In [5]:
# Prepare data with best n-gram config
train_pd = train_w.toPandas()
val_pd   = val.toPandas()
train_pd['ngram_text'] = extract_ngrams(train_pd['clean_text'], best_ngram)
val_pd['ngram_text']   = extract_ngrams(val_pd['clean_text'],   best_ngram)

train_best = spark.createDataFrame(
    train_pd[['ngram_text', 'sentiment', 'classWeight']].dropna()
).repartition(2)
val_best = spark.createDataFrame(
    val_pd[['ngram_text', 'sentiment']].dropna()
).repartition(2)

# Train helpfulness-weighted version too
train_pd_wh = train_wh.toPandas()
train_pd_wh['ngram_text'] = extract_ngrams(train_pd_wh['clean_text'], best_ngram)
train_best_wh = spark.createDataFrame(
    train_pd_wh[['ngram_text', 'sentiment', 'classWeight']].dropna()
).repartition(2)

stages_best = make_tfidf_pipeline('ngram_text')

reg_params   = [0.001, 0.003, 0.005, 0.01]
reg_results  = {}

for reg in reg_params:
    print(f'\nTesting regParam={reg}')
    t0 = time.time()
    try:
        lr = LogisticRegression(
            maxIter=200, regParam=reg, elasticNetParam=0.1,
            family='multinomial', weightCol='classWeight'
        )
        model = Pipeline(stages=stages_best + [lr]).fit(train_best)
        f1    = evaluator.evaluate(model.transform(val_best))
        reg_results[reg] = round(f1, 4)
        print(f'  Val F1: {f1:.4f} ({time.time()-t0:.0f}s)')
    except Exception as e:
        print(f'  Failed: {e}')
        reg_results[reg] = None

# Also test with helpfulness weighting
best_reg = max(reg_results, key=lambda k: reg_results[k] or 0)
print(f'\nTesting best regParam={best_reg} with helpfulness weighting...')
try:
    lr_wh = LogisticRegression(
        maxIter=200, regParam=best_reg, elasticNetParam=0.1,
        family='multinomial', weightCol='classWeight'
    )
    model_wh = Pipeline(stages=stages_best + [lr_wh]).fit(train_best_wh)
    f1_wh    = evaluator.evaluate(model_wh.transform(val_best))
    reg_results['helpfulness_weighted'] = round(f1_wh, 4)
    print(f'  Val F1 with helpfulness weighting: {f1_wh:.4f}')
except Exception as e:
    print(f'  Failed: {e}')

print('\n' + '='*50)
print(' REGPARAM GRID SEARCH RESULTS')
print('='*50)
print(f'{"RegParam":25s} | {"Val F1":>8s}')
print('-'*37)
for reg, score in reg_results.items():
    score_str = f'{score:.4f}' if score else 'FAILED'
    best_val  = max(v for v in reg_results.values() if v)
    marker    = ' <-- BEST' if score == best_val else ''
    print(f'{str(reg):25s} | {score_str:>8s}{marker}')



Testing regParam=0.001


  Val F1: 0.8497 (234s)

Testing regParam=0.003


  Val F1: 0.8560 (156s)

Testing regParam=0.005


  Val F1: 0.8563 (134s)

Testing regParam=0.01


  Val F1: 0.8547 (144s)

Testing best regParam=0.005 with helpfulness weighting...


  Val F1 with helpfulness weighting: 0.8556

 REGPARAM GRID SEARCH RESULTS
RegParam                  |   Val F1
-------------------------------------
0.001                     |   0.8497
0.003                     |   0.8560
0.005                     |   0.8563 <-- BEST
0.01                      |   0.8547
helpfulness_weighted      |   0.8556


## Cell 6 — Final Model Comparison

Using the best configuration found in Cells 4 and 5, we now train all three
classifier types and compare them head-to-head.

- LR uses the best n-gram config + best regParam + class weights
- SVC and NaiveBayes use balanced undersampling (no weightCol support)

**Expected runtime: 20-30 minutes.**

In [7]:
# Prepare balanced data with best ngram config
bal_pd = balanced_train.toPandas()
bal_pd['ngram_text'] = extract_ngrams(bal_pd['clean_text'], best_ngram)
train_bal = spark.createDataFrame(
    bal_pd[['ngram_text', 'sentiment']].dropna()
).repartition(2)

final_results = {}
final_models  = {}

def train_and_eval(name, classifier, train_df):
    print('\n' + '='*55)
    print(f' Training: {name}')
    print('='*55)
    t0 = time.time()
    try:
        pipeline = Pipeline(stages=stages_best + [classifier])
        model    = pipeline.fit(train_df)
        f1       = evaluator.evaluate(model.transform(val_best))
        final_results[name] = round(f1, 4)
        final_models[name]  = model
        print(f' Val F1: {f1:.4f} ({time.time()-t0:.0f}s)')
    except Exception as e:
        print(f' Failed: {e}')

# Best regParam from grid search
best_reg_val = max((v for v in reg_results.items() if isinstance(v[0], float)),
                   key=lambda x: x[1] or 0)[0]

# Model 1: Best LR configuration
lr_best = LogisticRegression(
    maxIter=300,
    regParam=best_reg_val,
    elasticNetParam=0.1,
    family='multinomial',
    weightCol='classWeight'
)
train_and_eval('LogisticRegression_Best', lr_best, train_best)

# Model 2: OneVsRest LinearSVC
svc = LinearSVC(maxIter=200, regParam=0.001)
ovr = OneVsRest(classifier=svc)
train_and_eval('OneVsRest_LinearSVC', ovr, train_bal)

# Model 3: Naive Bayes
nb = NaiveBayes(smoothing=1.0, modelType='multinomial')
train_and_eval('NaiveBayes', nb, train_bal)

print('\n' + '='*55)
print(' FINAL MODEL COMPARISON (Validation F1)')
print('='*55)
print(f'{"Model":35s} | {"Val F1":>8s}')
print('-'*47)
best_f1 = max(final_results.values())
for name, score in sorted(final_results.items(), key=lambda x: x[1], reverse=True):
    marker = ' <-- BEST' if score == best_f1 else ''
    print(f'{name:35s} | {score:.4f}{marker}')



 Training: LogisticRegression_Best


 Val F1: 0.8563 (125s)

 Training: OneVsRest_LinearSVC


 Val F1: 0.7629 (56s)

 Training: NaiveBayes


 Val F1: 0.7811 (17s)

 FINAL MODEL COMPARISON (Validation F1)
Model                               |   Val F1
-----------------------------------------------
LogisticRegression_Best             | 0.8563 <-- BEST
NaiveBayes                          | 0.7811
OneVsRest_LinearSVC                 | 0.7629


## Cell 7 — Full Test Evaluation on Best Model

The best model is evaluated on the held-out test set.
This data was never seen during training or model selection.

Metrics reported:
- Weighted F1: primary metric accounting for class sizes
- Accuracy: raw correct predictions
- Weighted Precision / Recall: quality per class
- Per-class recall: exposes which sentiment is hardest to detect
- Balanced accuracy: average recall across all classes equally weighted —
  the most honest metric for imbalanced multi-class problems

In [8]:
best_name  = max(final_results, key=final_results.get)
best_model = final_models[best_name]

# Prepare test set with best ngram config
test_pd = test.toPandas()
test_pd['ngram_text'] = extract_ngrams(test_pd['clean_text'], best_ngram)
test_best = spark.createDataFrame(
    test_pd[['ngram_text', 'sentiment', 'Id', 'ProductId', 'UserId', 'Time']].dropna()
).repartition(2)

test_preds = best_model.transform(test_best)

print(f'Best model : {best_name}')
print(f'Val F1     : {final_results[best_name]:.4f}')
print()
print('Test Set Metrics:')
print('-' * 40)
for metric in ['f1', 'accuracy', 'weightedPrecision', 'weightedRecall']:
    evaluator.setMetricName(metric)
    print(f'  {metric:22s} : {evaluator.evaluate(test_preds):.4f}')

per_class = (
    test_preds
    .groupBy('label')
    .agg(
        F.sum(F.when(F.col('label') == F.col('prediction'), 1).otherwise(0)).alias('tp'),
        F.count('*').alias('support')
    )
    .withColumn('recall', F.col('tp') / F.col('support'))
    .orderBy('label')
)

print('\nPer-class Recall:')
per_class.select('label', 'support', 'recall').show(truncate=False)

balanced_acc = per_class.select(F.avg('recall')).first()[0]
print(f'Balanced Accuracy (macro recall) : {balanced_acc:.4f}')
print()
print('Target: >0.65 good, >0.70 very good for 3-class imbalanced sentiment.')


Best model : LogisticRegression_Best
Val F1     : 0.8563

Test Set Metrics:
----------------------------------------


  f1                     : 0.8527


  accuracy               : 0.8397


  weightedPrecision      : 0.8715


  weightedRecall         : 0.8397

Per-class Recall:


+-----+-------+-------------------+
|label|support|recall             |
+-----+-------+-------------------+
|0.0  |30676  |0.8876646238101448 |
|1.0  |5707   |0.7615209391974768 |
|2.0  |2975   |0.49478991596638655|
+-----+-------+-------------------+



Balanced Accuracy (macro recall) : 0.7147

Target: >0.65 good, >0.70 very good for 3-class imbalanced sentiment.


## Cell 8 — Complete Results Summary Table

Prints a single comprehensive table showing all experiments performed.
This is the table to show during the presentation — it demonstrates
systematic experimentation, not random trial and error.

In [9]:
print('=' * 65)
print(' COMPLETE EXPERIMENT RESULTS SUMMARY')
print('=' * 65)

print('\nExperiment 1 — N-gram Configuration (LR, regParam=0.003)')
print('-' * 65)
print(f'{"Config":20s} | {"Val F1":>8s} | Notes')
print('-' * 65)
config_notes = {
    'unigram'    : 'Individual words only',
    'bigram'     : 'Word pairs only',
    'trigram'    : 'Word triplets only',
    'uni_bi'     : 'Words + pairs',
    'uni_bi_tri' : 'Words + pairs + triplets'
}
best_ng_f1 = max(v for v in ngram_results.values() if v)
for config in configs:
    score  = ngram_results.get(config)
    s_str  = f'{score:.4f}' if score else 'FAILED'
    marker = ' BEST' if score == best_ng_f1 else ''
    print(f'{config:20s} | {s_str:>8s} | {config_notes[config]}{marker}')

print(f'\nExperiment 2 — RegParam Grid Search (LR, config={best_ngram})')
print('-' * 65)
print(f'{"RegParam":25s} | {"Val F1":>8s} | Notes')
print('-' * 65)
reg_notes = {
    0.001 : 'Lightest regularisation',
    0.003 : 'Light regularisation',
    0.005 : 'Medium regularisation',
    0.01  : 'Standard regularisation',
    'helpfulness_weighted': 'Best reg + helpfulness boost'
}
best_reg_f1 = max(v for v in reg_results.values() if v)
for reg, score in reg_results.items():
    s_str  = f'{score:.4f}' if score else 'FAILED'
    marker = ' BEST' if score == best_reg_f1 else ''
    note   = reg_notes.get(reg, '')
    print(f'{str(reg):25s} | {s_str:>8s} | {note}{marker}')

print(f'\nExperiment 3 — Final Model Comparison (config={best_ngram}, regParam={best_reg_val})')
print('-' * 65)
print(f'{"Model":35s} | {"Val F1":>8s}')
print('-' * 65)
for name, score in sorted(final_results.items(), key=lambda x: x[1], reverse=True):
    marker = ' WINNER' if score == max(final_results.values()) else ''
    print(f'{name:35s} | {score:.4f}{marker}')

print('\n' + '=' * 65)
print(f' FINAL SELECTION: {best_name}')
print(f' Val F1          : {final_results[best_name]:.4f}')
print(f' N-gram config   : {best_ngram}')
print(f' RegParam        : {best_reg_val}')
print('=' * 65)

 COMPLETE EXPERIMENT RESULTS SUMMARY

Experiment 1 — N-gram Configuration (LR, regParam=0.003)
-----------------------------------------------------------------
Config               |   Val F1 | Notes
-----------------------------------------------------------------
unigram              |   0.8279 | Individual words only
bigram               |   0.8134 | Word pairs only
trigram              |   0.6609 | Word triplets only
uni_bi               |   0.8560 | Words + pairs BEST
uni_bi_tri           |   0.8494 | Words + pairs + triplets

Experiment 2 — RegParam Grid Search (LR, config=uni_bi)
-----------------------------------------------------------------
RegParam                  |   Val F1 | Notes
-----------------------------------------------------------------
0.001                     |   0.8497 | Lightest regularisation
0.003                     |   0.8560 | Light regularisation
0.005                     |   0.8563 | Medium regularisation BEST
0.01                      |   0.8547 | 

## Cell 9 — Save Best Model and Label Mapping

Saves the complete trained pipeline to disk using Spark native format.
The streaming consumer loads this model to make real-time predictions.

Also saves:
- `label_mapping.json` — maps numeric predictions (0,1,2) back to sentiment strings
- `model_info.json` — records which model won and with what configuration

In [10]:
MODEL_PATH = f'{BASE}/model/best_sentiment_model'

best_model.write().overwrite().save(MODEL_PATH)

# Save model metadata
meta = {
    'best_model'  : best_name,
    'val_f1'      : final_results[best_name],
    'ngram_config': best_ngram,
    'reg_param'   : best_reg_val,
    'model_path'  : MODEL_PATH
}
with open(f'{BASE}/model/model_info.json', 'w') as f:
    json.dump(meta, f, indent=2)

# Save label mapping — StringIndexer is always stage index 4
indexer_model = best_model.stages[4]
labels        = indexer_model.labels
label_mapping = {str(i): label for i, label in enumerate(labels)}

print('Label mapping (numeric prediction -> sentiment):')
for idx, label in label_mapping.items():
    count = next(r['count'] for r in label_counts if r['sentiment'] == label)
    print(f'  {idx} -> {label:10s} ({count:,} training samples)')

with open(f'{BASE}/model/label_mapping.json', 'w') as f:
    json.dump(label_mapping, f, indent=2)

print(f'\nModel saved  : {MODEL_PATH}')
print(f'Model type   : {best_name}')
print(f'Val F1       : {final_results[best_name]:.4f}')
print('\nTraining complete. Pipeline ready for streaming.')

Label mapping (numeric prediction -> sentiment):
  0 -> positive   (245,435 training samples)
  1 -> negative   (45,659 training samples)
  2 -> neutral    (23,806 training samples)

Model saved  : /Users/beethoven/BigData/AmazonReview/model/best_sentiment_model
Model type   : LogisticRegression_Best
Val F1       : 0.8563

Training complete. Pipeline ready for streaming.
